# Bronze — Statistics Iceland GDP Ingestion
Fetches quarterly GDP year-on-year growth from Statistics Iceland (Hagstofa Íslands) via the PX-Web REST API.

**Source:** Hagstofa Íslands PX-Web API  
**Dataset:** THJ01601 — Quarterly GDP (Mælikvarði=2 YoY growth, Skipting=14 Total GDP)  
**Output:** `bronze.statistics_iceland_raw` (Delta table)

In [ ]:
import requests
import pandas as pd

API_URL = "https://px.hagstofa.is/pxis/api/v1/is/Efnahagur/thjodhagsreikningar/landsframl/2_landsframleidsla_arsfj/THJ01601.px"
REBUILD_HISTORY = True 

if spark.catalog.tableExists("bronze.statistics_iceland_raw") and not REBUILD_HISTORY:
    is_full_load = False
    print("Incremental mode.")
else:
    is_full_load = True
    print("Full load mode.")

In [ ]:
payload = {
    "query": [
        {"code": "Mælikvarði", "selection": {"filter": "item", "values": ["2"]}},
        {"code": "Skipting",   "selection": {"filter": "item", "values": ["14"]}},
    ],
    "response": {"format": "json"},
}

response = requests.post(API_URL, json=payload)
data = response.json()

records = [
    {"quarter_raw": i["key"][2],
    "value_raw": i["values"][0],
    "ingested_at": pd.Timestamp.now()} for i in data["data"]
]
df_raw = pd.DataFrame(records)

In [ ]:
spark_df = spark.createDataFrame(df_raw)

if is_full_load:
    spark_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("bronze.statistics_iceland_raw")
    print("Full history rebuild complete.")
else:
    spark_df.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable("bronze.statistics_iceland_raw")
    print(f"Appended {spark_df.count()} records.")